In [2]:
!pip -q install duckdb pyarrow pandas

import duckdb
import pandas as pd

# Define File Paths and Create Database Connection

In [3]:
PARQUET_PATH = "/content/2026-01-13_hra-logs.parquet"
DB_PATH = "/content/hra_usage.duckdb"

con = duckdb.connect(DB_PATH)

## Building Base View from Raw Parquet

In this step, we create a reusable base view from the raw parquet file.  
We also combine the separate `date` and `time` columns into a single `event_ts` timestamp field, which will later help with time-based aggregation and trend analysis.


In [5]:
con.execute(f"""
CREATE OR REPLACE VIEW logs_base AS
SELECT
    anon_id,
    date,
    time,
    CAST(CAST(date AS VARCHAR) || ' ' || time AS TIMESTAMP) AS event_ts,
    x_edge_location,
    sc_bytes,
    cs_method,
    cs_uri_stem,
    sc_status,
    cs_referer,
    cs_user_agent,
    cs_uri_query,
    cs_cookie,
    x_edge_result_type,
    x_edge_request_id,
    x_host_header,
    cs_protocol,
    cs_bytes,
    time_taken,
    ssl_protocol,
    ssl_cipher,
    x_edge_response_result_type,
    cs_protocol_version,
    time_to_first_byte,
    x_edge_detailed_result_type,
    sc_content_type,
    sc_content_len,
    sc_range_start,
    sc_range_end,
    timestamp,
    timestamp_ms,
    c_country,
    query,
    traffic_type,
    referrer,
    airport,
    month,
    day,
    distribution,
    site,
    year
FROM read_parquet('{PARQUET_PATH}')
WHERE date IS NOT NULL
  AND time IS NOT NULL
""")

# Basic Dataset Validation

In [6]:
summary_df = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    MIN(event_ts) AS min_ts,
    MAX(event_ts) AS max_ts,
    COUNT(DISTINCT anon_id) AS unique_users,
    COUNT(DISTINCT cs_uri_stem) AS unique_paths,
    COUNT(DISTINCT c_country) AS unique_countries
FROM logs_base
""").df()

display(summary_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,min_ts,max_ts,unique_users,unique_paths,unique_countries
0,12779123,2023-06-05 15:25:20,2026-01-12 23:59:56,689865,108043,209


## Create Clean Working Table

Here, we build a cleaned version of the raw logs that keeps only the columns needed for analysis.  
We also standardize path and referrer fields and derive day, week, and month timestamp buckets for later aggregation.


In [7]:
con.execute("""
CREATE OR REPLACE TABLE logs_clean AS
SELECT
    anon_id,
    event_ts,
    DATE_TRUNC('day', event_ts) AS event_day,
    DATE_TRUNC('week', event_ts) AS event_week,
    DATE_TRUNC('month', event_ts) AS event_month,
    LOWER(TRIM(cs_uri_stem)) AS path,
    LOWER(TRIM(cs_uri_query)) AS raw_query_string,
    sc_status,
    c_country,
    traffic_type,
    LOWER(TRIM(referrer)) AS referrer,
    LOWER(TRIM(cs_referer)) AS cs_referer,
    query,
    distribution,
    site
FROM logs_base
WHERE anon_id IS NOT NULL
  AND event_ts IS NOT NULL
  AND cs_uri_stem IS NOT NULL
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Data Quality Check

This section performs a quick sanity check on the cleaned table.  
We verify how many rows remain after cleaning, how many unique users are present, and how many distinct paths exist. This helps confirm that the cleaning process did not accidentally remove too much data.

In [8]:
data_quality_check_df = con.execute("""
SELECT
    COUNT(*) AS rows_after_cleaning,
    COUNT(DISTINCT anon_id) AS unique_users,
    COUNT(DISTINCT path) AS unique_paths
FROM logs_clean
""").df()

display(data_quality_check_df)

,rows_after_cleaning,unique_users,unique_paths
0,12779123,689865,107410


## Explore Most Frequent Paths

At this stage, we inspect the most requested paths in the cleaned dataset.  
This gives us an initial sense of what kinds of endpoints dominate the traffic and helps us understand how HRA interfaces and APIs appear in the logs.


In [9]:
top_paths_df = con.execute("""
SELECT
    path,
    COUNT(*) AS hits
FROM logs_clean
GROUP BY 1
ORDER BY hits DESC
LIMIT 100
""").df()

display(top_paths_df)

,path,hits
0,/ui/ccf-eui/styles.css,1623508
1,/,529782
2,/api/v1/collisions,433800
3,/asct-b/bonemarrow-pelvis/v1.0,415171
4,/asct-b/bonemarrow-pelvis/v1.0/,308667
...,...,...
95,/api/v1/reference-organ-scene,15824
96,/hubmap-hra-api/v1/tissue-blocks,15638
97,/media/material-symbols-outlined-latin-fill-no...,15314
98,/assets/images/carousel_sennet.svg,14772


## Initial Interface Mapping Attempt

This section represents an early pass at assigning requests to RUI, EUI, CDE, and OTHER.  
It was useful for initial exploration, but the mapping is refined later in the notebook after filtering static assets and inspecting API/UI path families more carefully.


In [10]:
con.execute("""
CREATE OR REPLACE VIEW logs_labeled AS
SELECT
    *,
    CASE
        -- EUI
        WHEN path LIKE '/ui/ccf-eui%' THEN 'EUI'
        WHEN path LIKE '%reference-organ%' THEN 'EUI'
        WHEN path LIKE '%scene%' THEN 'EUI'

        -- RUI
        WHEN path LIKE '%tissue-block%' THEN 'RUI'
        WHEN path LIKE '%collision%' THEN 'RUI'

        -- CDE
        WHEN path LIKE '%cell-distance%' THEN 'CDE'
        WHEN path LIKE '%histogram%' THEN 'CDE'
        WHEN path LIKE '%violin%' THEN 'CDE'

        ELSE 'OTHER'
    END AS interface_guess
FROM logs_clean
""")

In [11]:
interface_summary_df = con.execute("""
SELECT
    interface_guess,
    COUNT(*) AS events,
    COUNT(DISTINCT anon_id) AS users
FROM logs_labeled
GROUP BY 1
ORDER BY events DESC
""").df()

display(interface_summary_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,interface_guess,events,users
0,OTHER,10322133,287974
1,EUI,1955015,437044
2,RUI,501416,14107
3,CDE,559,279


In [12]:
con.execute("""
CREATE OR REPLACE TABLE monthly_usage AS
SELECT
    event_month,
    interface_guess,
    COUNT(*) AS event_count,
    COUNT(DISTINCT anon_id) AS unique_users
FROM logs_labeled
WHERE event_month >= DATE '2023-10-01'
  AND event_month < DATE '2026-04-01'
GROUP BY 1, 2
ORDER BY 1, 2
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [13]:
monthly_usage_df = con.execute("""
SELECT *
FROM monthly_usage
ORDER BY event_month, interface_guess
""").df()

display(monthly_usage_df.head(30))

,event_month,interface_guess,event_count,unique_users
0,2023-10-01,EUI,392,44
1,2023-10-01,OTHER,59535,3784
2,2023-10-01,RUI,284,33
3,2023-11-01,EUI,3299,243
4,2023-11-01,OTHER,113765,5144
5,2023-11-01,RUI,2392,194
6,2023-12-01,EUI,3028,392
7,2023-12-01,OTHER,140963,6739
8,2023-12-01,RUI,2069,320
9,2024-01-01,EUI,7564,399


In [14]:
top_paths_by_interface_df = con.execute("""
SELECT
    interface_guess,
    path,
    COUNT(*) AS hits
FROM logs_labeled
WHERE interface_guess <> 'OTHER'
GROUP BY 1, 2
ORDER BY interface_guess, hits DESC
""").df()

display(top_paths_by_interface_df.head(50))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,interface_guess,path,hits
0,CDE,/assets/content/cell-distance-explorer-page/da...,318
1,CDE,/assets/content/cell-distance-explorer-page/im...,215
2,CDE,/cde/histogram.component.css.map,11
3,CDE,/ui/cde-ui/cell-distance-vis.png,3
4,CDE,/cell-distance-explorer,1
5,CDE,/assets/content/cell-distance-explorer-page/im...,1
6,CDE,/assets/content/cell-distance-explorer-page/im...,1
7,CDE,/assets/content/cell-distance-explorer-page/im...,1
8,CDE,/assets/content/cell-distance-explorer-page/im...,1
9,CDE,/assets/content/cell-distance-explorer-page/im...,1


## Remove Static Asset Traffic

Not all requests represent meaningful feature usage.  
In this section, we remove obvious static asset requests such as CSS, JavaScript, images, and fonts so the remaining data is more useful for behavioral analysis.


In [15]:
con.execute("""
CREATE OR REPLACE TABLE logs_actions AS
SELECT *
FROM logs_clean
WHERE path NOT LIKE '%.css'
  AND path NOT LIKE '%.js'
  AND path NOT LIKE '%.svg'
  AND path NOT LIKE '%.png'
  AND path NOT LIKE '%.jpg'
  AND path NOT LIKE '%.jpeg'
  AND path NOT LIKE '%.gif'
  AND path NOT LIKE '%.woff'
  AND path NOT LIKE '%.woff2'
  AND path NOT LIKE '%.ttf'
  AND path NOT LIKE '%.ico'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Inspect Action-Oriented Paths

After filtering out static resources, we inspect the remaining high-frequency paths.  
This gives us a better view of actual content, API, and interaction-related requests that may correspond to user behavior.


In [16]:
action_paths_df = con.execute("""
SELECT
    path,
    COUNT(*) AS hits
FROM logs_actions
GROUP BY 1
ORDER BY hits DESC
LIMIT 200
""").df()

display(action_paths_df)

,path,hits
0,/,529782
1,/api/v1/collisions,433800
2,/asct-b/bonemarrow-pelvis/v1.0,415171
3,/asct-b/bonemarrow-pelvis/v1.0/,308667
4,/sparql,284465
...,...,...
195,/digital-objects/omap/18-lymph-node-cell-dive-...,2362
196,/digital-objects/omap/15-intestine-codex/v1.0/...,2356
197,/digital-objects/omap/17-thymus-ibex/v1.0/asse...,2355
198,/digital-objects/omap/13-pancreas-codex/v1.1/a...,2349


## Remove Root Path for Feature Analysis

The root path `/` is useful for general traffic analysis, but it is not helpful for understanding specific feature usage.  
We remove it here so the later feature-level results are easier to interpret.


In [17]:
con.execute("""
CREATE OR REPLACE TABLE logs_actions_no_root AS
SELECT *
FROM logs_actions
WHERE path <> '/'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Inspect API Endpoints

This section isolates API-related paths from the filtered dataset.  
The goal is to understand which backend endpoints are most frequently used and to identify endpoint families that may correspond to RUI, EUI, or CDE workflows.


In [18]:
api_paths_df = con.execute("""
SELECT
    path,
    COUNT(*) AS hits
FROM logs_actions_no_root
WHERE path LIKE '/api/%'
   OR path LIKE '/hra-api%'
   OR path LIKE '/hubmap-hra-api%'
GROUP BY 1
ORDER BY hits DESC
LIMIT 200
""").df()

display(api_paths_df)

,path,hits
0,/api/v1/collisions,433800
1,/api/v1/extraction-site,200121
2,/hra-api/v1/get-spatial-placement,88278
3,/api/,74691
4,/api/v1/db-status,62690
...,...,...
195,/api/v1/ds-graph,132
196,/api/shared/config.env,128
197,/api/anlei/register,127
198,/hra-api--staging/v1/sennet/rui_locations.jsonld,124


## Inspect UI Routes

Here, we examine frontend interface routes under `/ui/`.  
These paths are especially useful for identifying which requests belong to EUI, RUI, and CDE, and they help improve the interface-mapping logic.


In [19]:
ui_paths_df = con.execute("""
SELECT
    path,
    COUNT(*) AS hits
FROM logs_actions_no_root
WHERE path LIKE '/ui/%'
GROUP BY 1
ORDER BY hits DESC
LIMIT 200
""").df()

display(ui_paths_df)

,path,hits
0,/ui/ftu-ui/assets/temp/ftu-cell-summaries.jsonld,4551
1,/ui/ftu-ui/assets/temp/ftu-datasets.jsonld,4503
2,/ui/ftu-ui/assets/temp/2d-ftu-illustrations.js...,3970
3,/ui/cde-ui/assets/data/landing-page/cards.json,3203
4,/ui/ftu-ui/assets/resources.yml,2127
...,...,...
195,/ui/asctb-reporter/t/,2
196,/ui/ftu-ui-small-wc/$%257bt%257d,2
197,/ui/ccf-rui/index.html,2
198,/ui/ftu-ui-small-wc/%27+t+%27,2


## Refined Interface Mapping

Using the patterns observed in the API and UI path exploration steps, we now rebuild the interface mapping.  
Each request is assigned to one of the target interfaces: RUI, EUI, CDE, or OTHER. This refined version is more reliable than the earlier exploratory pass.


In [22]:
con.execute("""
CREATE OR REPLACE VIEW logs_labeled AS
SELECT
    *,
    CASE
        -- Frontend UI routes
        WHEN path LIKE '/ui/ccf-eui%' THEN 'EUI'
        WHEN path LIKE '/ui/ccf-rui%' THEN 'RUI'
        WHEN path LIKE '/ui/cde-ui%' THEN 'CDE'
        WHEN path LIKE '/ui/ftu-ui%' THEN 'OTHER'

        -- RUI-related API endpoints
        WHEN path LIKE '%collision%' THEN 'RUI'
        WHEN path LIKE '%tissue-block%' THEN 'RUI'
        WHEN path LIKE '%get-spatial-placement%' THEN 'RUI'
        WHEN path LIKE '%extraction-site%' THEN 'RUI'
        WHEN path LIKE '%rui_locations%' THEN 'RUI'

        -- EUI-related API/content endpoints
        WHEN path LIKE '%reference-organ%' THEN 'EUI'
        WHEN path LIKE '%scene%' THEN 'EUI'
        WHEN path LIKE '%ontology%' THEN 'EUI'
        WHEN path LIKE '%cell-type%' THEN 'EUI'
        WHEN path LIKE '%biomarker%' THEN 'EUI'
        WHEN path LIKE '%provider-names%' THEN 'EUI'
        WHEN path LIKE '%technology-names%' THEN 'EUI'
        WHEN path LIKE '%aggregate-results%' THEN 'EUI'
        WHEN path LIKE '%ontology-tree-model%' THEN 'EUI'
        WHEN path LIKE '%cell-type-tree-model%' THEN 'EUI'

        -- CDE-related API/content endpoints
        WHEN path LIKE '%cde%' THEN 'CDE'
        WHEN path LIKE '%cell-distance%' THEN 'CDE'
        WHEN path LIKE '%histogram%' THEN 'CDE'
        WHEN path LIKE '%violin%' THEN 'CDE'

        ELSE 'OTHER'
    END AS interface_guess
FROM logs_actions_no_root
""")

> **Note:** Interface labels in this notebook are inferred from path patterns and endpoint families observed in the logs.  
> They are useful for analysis, but they should be interpreted as heuristic mappings rather than perfect ground-truth labels.


In [23]:
interface_summary_df = con.execute("""
SELECT
    interface_guess,
    COUNT(*) AS events,
    COUNT(DISTINCT anon_id) AS users
FROM logs_labeled
GROUP BY 1
ORDER BY events DESC
""").df()

display(interface_summary_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,interface_guess,events,users
0,OTHER,5572832,216037
1,RUI,830344,17723
2,EUI,451053,25603
3,CDE,16348,4800


## Inspect Top Paths by Interface

Here, we review the most frequent paths within each mapped interface.  
This helps validate the quality of the refined mapping and gives a first look at the kinds of requests associated with RUI, EUI, and CDE.


In [24]:
top_paths_by_interface_df = con.execute("""
SELECT
    interface_guess,
    path,
    COUNT(*) AS hits
FROM logs_labeled
WHERE interface_guess <> 'OTHER'
GROUP BY 1, 2
ORDER BY interface_guess, hits DESC
""").df()

display(top_paths_by_interface_df.head(100))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,interface_guess,path,hits
0,CDE,/cde/,6276
1,CDE,/ui/cde-ui/assets/data/landing-page/cards.json,3203
2,CDE,/cde,2045
3,CDE,/ui/cde-ui/assets/data/examples/index.json,1333
4,CDE,/cde/example/0,444
...,...,...,...
95,CDE,/hra-kg--staging/graph/hra-pop/v0.11.1/assets/...,2
96,CDE,/hra-kg--staging/graph/hra-pop/v1.0/assets/cor...,2
97,CDE,/hra-kg--staging/graph/hra-pop/v0.11.1/assets/...,2
98,CDE,/hra-kg--staging/graph/hra-pop/v1.0/assets/cor...,2


## Initial Summary Tables from Refined Interface Mapping

At this stage, we build summary tables directly from the refined interface-labeled data.  
These tables are useful for early validation, but the feature and trend summaries are further cleaned later by removing remaining non-feature resource requests.


In [25]:
con.execute("""
CREATE OR REPLACE TABLE monthly_usage AS
SELECT
    event_month,
    interface_guess,
    COUNT(*) AS event_count,
    COUNT(DISTINCT anon_id) AS unique_users
FROM logs_labeled
WHERE event_month >= DATE '2023-10-01'
  AND event_month < DATE '2026-04-01'
GROUP BY 1, 2
ORDER BY 1, 2
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [26]:
monthly_usage_df = con.execute("""
SELECT *
FROM monthly_usage
ORDER BY event_month, interface_guess
""").df()

display(monthly_usage_df.head(50))

,event_month,interface_guess,event_count,unique_users
0,2023-10-01,EUI,2315,134
1,2023-10-01,OTHER,20586,2664
2,2023-10-01,RUI,307,34
3,2023-11-01,EUI,16417,342
4,2023-11-01,OTHER,48693,4006
5,2023-11-01,RUI,15347,218
6,2023-12-01,EUI,12806,482
7,2023-12-01,OTHER,54688,5544
8,2023-12-01,RUI,31500,351
9,2024-01-01,CDE,1,1


In [27]:
con.execute("""
CREATE OR REPLACE TABLE feature_frequency AS
SELECT
    interface_guess,
    path AS feature_proxy,
    COUNT(*) AS event_count,
    COUNT(DISTINCT anon_id) AS unique_users
FROM logs_labeled
GROUP BY 1, 2
ORDER BY event_count DESC
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [28]:
top_features_df = con.execute("""
SELECT *
FROM feature_frequency
WHERE interface_guess <> 'OTHER'
ORDER BY event_count DESC
LIMIT 50
""").df()

display(top_features_df)

,interface_guess,feature_proxy,event_count,unique_users
0,RUI,/api/v1/collisions,433800,1693
1,RUI,/api/v1/extraction-site,200121,214
2,RUI,/hra-api/v1/get-spatial-placement,88278,23
3,EUI,/api/v1/cell-type-term-occurences,40267,6833
4,RUI,/api/v1/tissue-blocks,39975,10034
5,EUI,/api/v1/aggregate-results,34649,9954
6,EUI,/api/v1/ontology-term-occurences,29283,6825
7,EUI,/api/v1/scene,29158,7048
8,EUI,/api/v1/biomarker-term-occurences,26527,6831
9,EUI,/api/v1/ontology-tree-model,25687,11667


In [29]:
con.execute("""
CREATE OR REPLACE TABLE traffic_summary AS
SELECT
    traffic_type,
    COUNT(*) AS events,
    COUNT(DISTINCT anon_id) AS users
FROM logs_labeled
GROUP BY 1
ORDER BY events DESC
""")

In [30]:
con.execute("""
CREATE OR REPLACE TABLE geography_summary AS
SELECT
    c_country,
    COUNT(*) AS events,
    COUNT(DISTINCT anon_id) AS users
FROM logs_labeled
GROUP BY 1
ORDER BY events DESC
""")

## Create Feature-Focused Action Table

Even after removing static assets, some requests still reflect background resources rather than meaningful feature usage.  
In this section, we create a stricter filtered table intended for feature analysis by excluding additional resource files and system-level endpoints.


In [31]:
con.execute("""
CREATE OR REPLACE TABLE logs_feature_actions AS
SELECT *
FROM logs_labeled
WHERE path NOT LIKE '%/assets/%'
  AND path NOT LIKE '%.json'
  AND path NOT LIKE '%.jsonld'
  AND path NOT LIKE '%.yml'
  AND path NOT LIKE '%db-status%'
  AND path NOT LIKE '%config.env%'
  AND path NOT LIKE '/api'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Compute Feature Frequency

This table summarizes how often each endpoint-like feature proxy appears in the filtered dataset.  
Because the logs are request-based, these results should be interpreted as feature-related endpoint usage rather than exact user click counts.


In [32]:
con.execute("""
CREATE OR REPLACE TABLE feature_frequency AS
SELECT
    interface_guess,
    path AS feature_proxy,
    COUNT(*) AS event_count,
    COUNT(DISTINCT anon_id) AS unique_users
FROM logs_feature_actions
GROUP BY 1, 2
ORDER BY event_count DESC
""")

> **Note:** The feature frequency results are based on endpoint-level request patterns.  
> They should be interpreted as feature-related usage proxies, not exact clickstream actions.


In [33]:
top_features_df = con.execute("""
SELECT *
FROM feature_frequency
WHERE interface_guess <> 'OTHER'
ORDER BY event_count DESC
LIMIT 50
""").df()

display(top_features_df)

,interface_guess,feature_proxy,event_count,unique_users
0,RUI,/api/v1/collisions,433800,1693
1,RUI,/api/v1/extraction-site,200121,214
2,RUI,/hra-api/v1/get-spatial-placement,88278,23
3,EUI,/api/v1/cell-type-term-occurences,40267,6833
4,RUI,/api/v1/tissue-blocks,39975,10034
5,EUI,/api/v1/aggregate-results,34649,9954
6,EUI,/api/v1/ontology-term-occurences,29283,6825
7,EUI,/api/v1/scene,29158,7048
8,EUI,/api/v1/biomarker-term-occurences,26527,6831
9,EUI,/api/v1/ontology-tree-model,25687,11667


## Inspect Top and Least Used Features

In this section, we identify the most frequently used and least frequently used feature proxies across interfaces.  
This supports the project goal of understanding which parts of the Human Atlas system are heavily used and which are rarely accessed.


In [34]:
least_features_df = con.execute("""
SELECT *
FROM feature_frequency
WHERE interface_guess <> 'OTHER'
  AND event_count >= 5
ORDER BY event_count ASC
LIMIT 50
""").df()

display(least_features_df)

,interface_guess,feature_proxy,event_count,unique_users
0,CDE,"/ui/cde-ui/$%257bnew%2520url(t(c),n%7c%7clocat...",5,5
1,RUI,/ui/ccf-rui/styles.css.map,5,2
2,CDE,"/ui/cde-ui/t,document.baseuri",5,5
3,RUI,/hra-kg--staging/landmark/heart-female-extract...,5,4
4,RUI,/ui/ccf-rui/wc.js:2:609/n,5,5
5,RUI,/hra-kg--staging/landmark/heart-male-extractio...,5,4
6,CDE,/cde/visualization.component.css.map,5,1
7,RUI,/ui/ccf-rui/%27+(e=r)+%27,5,5
8,CDE,/ui/cde-ui/$%257br%257d,5,5
9,RUI,/hra-kg--staging/landmark/spleen-female-extrac...,5,5


## Build Monthly Usage Table

Here, we aggregate usage by month and interface using the stricter feature-focused dataset.  
This table supports temporal trend analysis and helps compare how activity changes over time across RUI, EUI, and CDE.


In [35]:
con.execute("""
CREATE OR REPLACE TABLE monthly_usage AS
SELECT
    event_month,
    interface_guess,
    COUNT(*) AS event_count,
    COUNT(DISTINCT anon_id) AS unique_users
FROM logs_feature_actions
WHERE event_month >= DATE '2023-10-01'
  AND event_month < DATE '2026-04-01'
GROUP BY 1, 2
ORDER BY 1, 2
""")

## Preview Monthly Usage Trends

This step displays the monthly aggregated usage table so we can quickly verify that the trend data looks correct before exporting it for visualization and reporting.


In [36]:
monthly_usage_df = con.execute("""
SELECT *
FROM monthly_usage
ORDER BY event_month, interface_guess
""").df()

display(monthly_usage_df.head(50))

,event_month,interface_guess,event_count,unique_users
0,2023-10-01,EUI,2233,76
1,2023-10-01,OTHER,17573,2219
2,2023-10-01,RUI,286,34
3,2023-11-01,EUI,16369,318
4,2023-11-01,OTHER,43910,3501
5,2023-11-01,RUI,15015,199
6,2023-12-01,EUI,12754,458
7,2023-12-01,OTHER,49879,5022
8,2023-12-01,RUI,31316,332
9,2024-01-01,CDE,1,1


## Summarize Traffic Type

This section groups requests by `traffic_type` to compare human and bot activity.  
It supports the project’s bot-vs-human analysis and helps us understand what proportion of total traffic likely comes from real users.


In [37]:
traffic_summary_df = con.execute("""
SELECT
    traffic_type,
    COUNT(*) AS events,
    COUNT(DISTINCT anon_id) AS users
FROM logs_labeled
GROUP BY 1
ORDER BY events DESC
""").df()

display(traffic_summary_df)

,traffic_type,events,users
0,Likely Human,4305153,176421
1,Bot,2236420,39770
2,AI-Assistant / Bot,329004,5388


## Summarize Geographic Distribution

Here, we aggregate usage by country using the `c_country` field.  
This helps answer the geography-related project question by showing which countries generate the most activity in the dataset.


In [38]:
geography_summary_df = con.execute("""
SELECT
    c_country,
    COUNT(*) AS events,
    COUNT(DISTINCT anon_id) AS users
FROM logs_labeled
GROUP BY 1
ORDER BY events DESC
LIMIT 50
""").df()

display(geography_summary_df)

,c_country,events,users
0,US,5132708,94343
1,SG,273257,27646
2,HK,185376,3534
3,DE,176484,6317
4,GB,140493,7953
5,FR,130761,4046
6,JP,112692,2559
7,CN,76295,17932
8,IE,62820,1096
9,EC,58345,771


## Export Final Tables

In this step, we export the cleaned summary tables as CSV files.


In [39]:
con.execute("COPY monthly_usage TO '/content/monthly_usage.csv' (HEADER, DELIMITER ',')")
con.execute("COPY feature_frequency TO '/content/feature_frequency.csv' (HEADER, DELIMITER ',')")
con.execute("COPY traffic_summary TO '/content/traffic_summary.csv' (HEADER, DELIMITER ',')")
con.execute("COPY geography_summary TO '/content/geography_summary.csv' (HEADER, DELIMITER ',')")